In [1]:
import numpy as np
import networkx as nx
import polars as pl
from collections import defaultdict
from tqdm.notebook import tqdm
from node2vec import Node2Vec

In [2]:
articles_path='../data/articles.parquet'
transaction_path='../data/transactions.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [4]:
def build_item_graph(transactions,topk_neighbour,max_hist=30):
    edge_cnt=defaultdict(float)
    for _,g in tqdm(transactions.group_by('customer_id'),total=transactions['customer_id'].unique().shape[0]):
        g=g.sort('time',descending=True).head(max_hist)
        items=g['article_id'].to_list()

        for i in range(len(items)):
            for j in range(i+1,len(items)):
                a,b=items[i],items[j]
                if a==b:
                    continue
                edge_cnt[(a,b)]+=1
                edge_cnt[(b,a)]+=1
    item_edges=defaultdict(list)
    for (i,j),w in edge_cnt.items():
        item_edges[i].append((w,j))
    G=nx.Graph() # 建立无向图
    for i,js in item_edges.items():
        js=sorted(js,reverse=True)[:topk_neighbour]
        for w,j in js:
            G.add_edge(str(i),str(j),weight=w)
    return G

In [ ]:
G=build_item_graph(transactions,topk_neighbour=50)
node2vec = Node2Vec(
    G,
    dimensions=64, #最终得到的embedding长度
    walk_length=20, #每一次随机游走的长度
    num_walks=10, #每个节点发起多少次随机游走
    workers=8,
    weight_key='weight', # 以那个字段作为边权
    p=1, # 控制回退概率，<1更容易回头，>1不容易回头，=1不偏不倚
    q=1, # 是否向远处扩散
)
model = node2vec.fit(
    window=5,
    min_count=1,
    sg=1,
    epochs=5
)

  0%|          | 0/1362281 [00:00<?, ?it/s]